# Metric Source Reconciliation V1

Purpose: reconcile the `sequence_risk_action_cap_screen_v1` blocker where official qlib-style relative return disagreed with the existing `evaluate_split`-style gate. This is a diagnostic/model-selection notebook, not a promotion notebook.

Governance:
- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sequence_risk_action_cap_screen_v1`.
- **Role**: `diagnose`.
- **Target window**: `wf_2022` and `wf_2023_2024`; accepted trade-off is no promotion until metric alignment is explicit and fixed-split / `wf_2020` are checked.
- **Changed layer**: `model_selection`, `harness_metric_diagnostic`, `src_capability`.
- **External reference**: qlib backtest semantics and prior qlib/PPO/safe-RL/PBO references: https://github.com/microsoft/qlib, https://arxiv.org/abs/2009.11189, https://arxiv.org/abs/1707.06347, https://arxiv.org/abs/1712.06924, https://arxiv.org/abs/2006.04779, https://doi.org/10.21314/JCF.2016.322.
- **Hypothesis**: the apparent disagreement is caused by benchmark-window alignment. Isolated qlib fold backtests start the account at the fold boundary, but the policy cannot trade for the first `LOOKBACK` rows. Calendar buy-and-hold includes that warmup interval; `evaluate_split` benchmarks from the first tradable row.
- **Acceptance criteria**: produce side-by-side calendar vs tradable relative metrics, identify any gate flips, keep promotion blocked if a gate flip is warmup-driven.
- **Failure mode**: if metric variants do not explain the disagreement, do not tune action-cap thresholds; inspect actor replay / portfolio accounting next.
- **Comparison-table row key**: `metric_source_reconciliation_v1`.

Historical duplicate check:
- Closest success case: `sequence_risk_action_cap_screen_v1` reduced `wf_2022` MaxDD under 15% while leaving `wf_2023_2024` execution unchanged for the selected candidate.
- Closest failure case: earlier reward PPO improved return while worsening sequence-state exposure and drawdown.
- This run is not a duplicate because it changes only metric attribution and reusable metric helper code, not policy, reward, actor training, or action-cap thresholds.


In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if ROOT.name != 'workspace':
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.core.metric_reconciliation import compute_relative_metric_variants
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH, load_profile

EXP = 'metric_source_reconciliation_v1'
OUT_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXP
SOURCE_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / 'sequence_risk_action_cap_screen_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SELECTED_CANDIDATE = 'seq_cap10_two_of_three'
FOLD_WINDOWS = {
    'wf_2022': ('2022-01-01', '2022-12-31'),
    'wf_2023_2024': ('2024-01-01', '2024-12-31'),
}

print(f'ROOT={ROOT}')
print(f'LOOKBACK={LOOKBACK}')


ROOT=/Users/rziiiii/.openclaw/workspace
LOOKBACK=60


In [2]:
action_metrics = pd.read_csv(SOURCE_DIR / 'action_cap_metrics.csv')
source_selection = json.loads((SOURCE_DIR / 'selection.json').read_text())
profile = load_profile(PROFILE_NAME)

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=profile.feature.packs, profile_name=profile.name)
price = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()

base_cols = [
    'candidate', 'fold', 'return', 'max_dd', 'buy_and_hold_return', 'relative_return',
    'effective_start', 'effective_end', 'trace_target_cap_changed_days', 'trace_overlay_ready_days',
]
action_metrics[base_cols].sort_values(['fold', 'candidate'])


[61362:MainThread](2026-05-26 11:38:30,851) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[61362:MainThread](2026-05-26 11:38:31,605) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[61362:MainThread](2026-05-26 11:38:31,605) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[61362:MainThread](2026-05-26 11:38:34,403) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[61362:MainThread](2026-05-26 11:38:34,406) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[61362:MainThread](2026-05-26 11:38:34,407) INFO - qlib.timer - [log.py:127] - Time cost: 0.006s | Init data Done


,candidate,fold,return,max_dd,buy_and_hold_return,relative_return,effective_start,effective_end,trace_target_cap_changed_days,trace_overlay_ready_days
0,baseline_no_action_cap,wf_2022,-16.534504,18.396429,-86.588072,70.053568,2022-01-03,2022-12-29,0,0
4,seq_cap10_any,wf_2022,-7.725714,11.530076,-86.588072,78.862358,2022-01-03,2022-12-29,104,188
1,seq_cap10_two_of_three,wf_2022,-7.725714,11.530076,-86.588072,78.862358,2022-01-03,2022-12-29,104,183
2,seq_cap15_two_of_three,wf_2022,-9.884809,15.157706,-86.588072,76.703263,2022-01-03,2022-12-29,104,183
3,seq_cap20_two_of_three,wf_2022,-9.500257,17.330761,-86.588072,77.087814,2022-01-03,2022-12-29,43,183
5,baseline_no_action_cap,wf_2023_2024,-5.811867,13.472617,-0.356635,-5.455232,2024-01-02,2024-12-30,0,0
9,seq_cap10_any,wf_2023_2024,-6.322983,9.733743,-0.356635,-5.966348,2024-01-02,2024-12-30,41,54
6,seq_cap10_two_of_three,wf_2023_2024,-5.811867,13.472617,-0.356635,-5.455232,2024-01-02,2024-12-30,0,0
7,seq_cap15_two_of_three,wf_2023_2024,-5.811867,13.472617,-0.356635,-5.455232,2024-01-02,2024-12-30,0,0
8,seq_cap20_two_of_three,wf_2023_2024,-5.811867,13.472617,-0.356635,-5.455232,2024-01-02,2024-12-30,0,0


In [3]:
rows = []
for row in action_metrics.to_dict('records'):
    fold = row['fold']
    requested_seg = FOLD_WINDOWS[fold]
    effective_seg = (str(row['effective_start']), str(row['effective_end']))
    variants = compute_relative_metric_variants(
        strategy_return=float(row['return']),
        price_series=price,
        requested_seg=requested_seg,
        effective_seg=effective_seg,
        lookback=LOOKBACK,
        target_relative_return=TARGET_RELATIVE_RETURN,
    )
    rows.append({
        'candidate': row['candidate'],
        'fold': fold,
        'return': float(row['return']),
        'max_dd': float(row['max_dd']),
        'avg_pos': float(row.get('avg_pos', math.nan)),
        'source_buy_and_hold_return': float(row['buy_and_hold_return']),
        'source_relative_return': float(row['relative_return']),
        'trace_target_cap_changed_days': int(row.get('trace_target_cap_changed_days', 0)),
        'trace_overlay_ready_days': int(row.get('trace_overlay_ready_days', 0)),
        **variants,
        'maxdd_gate': bool(float(row['max_dd']) <= TARGET_MAX_DD),
    })

recon = pd.DataFrame(rows)
recon['source_matches_calendar_bnh'] = (recon['source_buy_and_hold_return'] - recon['calendar_buy_and_hold_return']).abs() < 1e-9
recon['source_matches_calendar_relative'] = (recon['source_relative_return'] - recon['calendar_relative_return']).abs() < 1e-9
recon['calendar_full_gate'] = recon['calendar_gate'] & recon['maxdd_gate']
recon['tradable_full_gate'] = recon['tradable_gate'] & recon['maxdd_gate']
recon['full_gate_flip'] = recon['calendar_full_gate'] != recon['tradable_full_gate']
recon_path = OUT_DIR / 'metric_reconciliation.csv'
recon.to_csv(recon_path, index=False)

recon[[
    'candidate', 'fold', 'return', 'max_dd',
    'calendar_buy_and_hold_return', 'tradable_buy_and_hold_return',
    'calendar_relative_return', 'tradable_relative_return',
    'relative_metric_gap', 'calendar_full_gate', 'tradable_full_gate',
    'full_gate_flip', 'mismatch_reason', 'first_tradable_date', 'warmup_days',
    'trace_target_cap_changed_days',
]].sort_values(['fold', 'candidate'])


,candidate,fold,return,max_dd,calendar_buy_and_hold_return,tradable_buy_and_hold_return,calendar_relative_return,tradable_relative_return,relative_metric_gap,calendar_full_gate,tradable_full_gate,full_gate_flip,mismatch_reason,first_tradable_date,warmup_days,trace_target_cap_changed_days
0,baseline_no_action_cap,wf_2022,-16.534504,18.396429,-86.588072,-76.614267,70.053568,60.079764,-9.973804,False,False,False,material_warmup_benchmark_alignment_gap,2022-03-30,60,0
4,seq_cap10_any,wf_2022,-7.725714,11.530076,-86.588072,-76.614267,78.862358,68.888553,-9.973804,True,True,False,material_warmup_benchmark_alignment_gap,2022-03-30,60,104
1,seq_cap10_two_of_three,wf_2022,-7.725714,11.530076,-86.588072,-76.614267,78.862358,68.888553,-9.973804,True,True,False,material_warmup_benchmark_alignment_gap,2022-03-30,60,104
2,seq_cap15_two_of_three,wf_2022,-9.884809,15.157706,-86.588072,-76.614267,76.703263,66.729459,-9.973804,False,False,False,material_warmup_benchmark_alignment_gap,2022-03-30,60,104
3,seq_cap20_two_of_three,wf_2022,-9.500257,17.330761,-86.588072,-76.614267,77.087814,67.114010,-9.973804,False,False,False,material_warmup_benchmark_alignment_gap,2022-03-30,60,43
5,baseline_no_action_cap,wf_2023_2024,-5.811867,13.472617,-0.356635,-39.952716,-5.455232,34.140849,39.596081,False,True,True,warmup_benchmark_alignment_gate_flip,2024-03-28,60,0
9,seq_cap10_any,wf_2023_2024,-6.322983,9.733743,-0.356635,-39.952716,-5.966348,33.629733,39.596081,False,True,True,warmup_benchmark_alignment_gate_flip,2024-03-28,60,41
6,seq_cap10_two_of_three,wf_2023_2024,-5.811867,13.472617,-0.356635,-39.952716,-5.455232,34.140849,39.596081,False,True,True,warmup_benchmark_alignment_gate_flip,2024-03-28,60,0
7,seq_cap15_two_of_three,wf_2023_2024,-5.811867,13.472617,-0.356635,-39.952716,-5.455232,34.140849,39.596081,False,True,True,warmup_benchmark_alignment_gate_flip,2024-03-28,60,0
8,seq_cap20_two_of_three,wf_2023_2024,-5.811867,13.472617,-0.356635,-39.952716,-5.455232,34.140849,39.596081,False,True,True,warmup_benchmark_alignment_gate_flip,2024-03-28,60,0


In [4]:
candidate_rows = []
for candidate, group in recon.groupby('candidate', sort=False):
    selected_group = group.sort_values('fold')
    candidate_rows.append({
        'candidate': candidate,
        'min_calendar_relative_return': float(selected_group['calendar_relative_return'].min()),
        'min_tradable_relative_return': float(selected_group['tradable_relative_return'].min()),
        'max_max_dd': float(selected_group['max_dd'].max()),
        'calendar_full_gate_all': bool(selected_group['calendar_full_gate'].all()),
        'tradable_full_gate_all': bool(selected_group['tradable_full_gate'].all()),
        'full_gate_flip_any': bool(selected_group['full_gate_flip'].any()),
        'target_cap_changed_days_total': int(selected_group['trace_target_cap_changed_days'].sum()),
        'wf_2022_tradable_relative_return': float(selected_group[selected_group['fold'].eq('wf_2022')]['tradable_relative_return'].iloc[0]) if 'wf_2022' in set(selected_group['fold']) else math.nan,
        'wf_2023_2024_tradable_relative_return': float(selected_group[selected_group['fold'].eq('wf_2023_2024')]['tradable_relative_return'].iloc[0]) if 'wf_2023_2024' in set(selected_group['fold']) else math.nan,
        'wf_2022_max_dd': float(selected_group[selected_group['fold'].eq('wf_2022')]['max_dd'].iloc[0]) if 'wf_2022' in set(selected_group['fold']) else math.nan,
        'wf_2023_2024_max_dd': float(selected_group[selected_group['fold'].eq('wf_2023_2024')]['max_dd'].iloc[0]) if 'wf_2023_2024' in set(selected_group['fold']) else math.nan,
    })

candidate_summary = pd.DataFrame(candidate_rows).sort_values(
    ['tradable_full_gate_all', 'max_max_dd', 'target_cap_changed_days_total'],
    ascending=[False, True, True],
).reset_index(drop=True)
candidate_summary_path = OUT_DIR / 'candidate_reconciliation_summary.csv'
candidate_summary.to_csv(candidate_summary_path, index=False)

selected_recon = recon[recon['candidate'].eq(SELECTED_CANDIDATE)].sort_values('fold').reset_index(drop=True)
selected_summary = candidate_summary[candidate_summary['candidate'].eq(SELECTED_CANDIDATE)].iloc[0].to_dict()

candidate_summary


,candidate,min_calendar_relative_return,min_tradable_relative_return,max_max_dd,calendar_full_gate_all,tradable_full_gate_all,full_gate_flip_any,target_cap_changed_days_total,wf_2022_tradable_relative_return,wf_2023_2024_tradable_relative_return,wf_2022_max_dd,wf_2023_2024_max_dd
0,seq_cap10_any,-5.966348,33.629733,11.530076,False,True,True,145,68.888553,33.629733,11.530076,9.733743
1,seq_cap10_two_of_three,-5.455232,34.140849,13.472617,False,True,True,104,68.888553,34.140849,11.530076,13.472617
2,seq_cap15_two_of_three,-5.455232,34.140849,15.157706,False,False,True,104,66.729459,34.140849,15.157706,13.472617
3,seq_cap20_two_of_three,-5.455232,34.140849,17.330761,False,False,True,43,67.114010,34.140849,17.330761,13.472617
4,baseline_no_action_cap,-5.455232,34.140849,18.396429,False,False,True,0,60.079764,34.140849,18.396429,13.472617


In [5]:
gate_flip_rows = recon[recon['full_gate_flip']].sort_values(['candidate', 'fold'])
root_cause = 'warmup_benchmark_alignment_gate_flip' if not gate_flip_rows.empty else 'no_gate_flip_detected'
selected_two_fold_tradable_gate = bool(selected_recon['tradable_full_gate'].all())
selected_two_fold_calendar_gate = bool(selected_recon['calendar_full_gate'].all())

selection = {
    'experiment': EXP,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'role': 'diagnose',
    'canonical_baseline': source_selection.get('canonical_baseline'),
    'direct_comparison_baseline': 'sequence_risk_action_cap_screen_v1',
    'profile': PROFILE_NAME,
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'decision': 'metric_source_reconciled_warmup_alignment_no_promotion',
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'root_cause': root_cause,
    'selected_candidate': SELECTED_CANDIDATE,
    'selected_calendar_full_gate_all': selected_two_fold_calendar_gate,
    'selected_tradable_full_gate_all': selected_two_fold_tradable_gate,
    'selected_min_calendar_relative_return': float(selected_summary['min_calendar_relative_return']),
    'selected_min_tradable_relative_return': float(selected_summary['min_tradable_relative_return']),
    'selected_max_max_dd': float(selected_summary['max_max_dd']),
    'selected_target_cap_changed_days_total': int(selected_summary['target_cap_changed_days_total']),
    'gate_flip_count': int(recon['full_gate_flip'].sum()),
    'gate_flip_candidates': sorted(gate_flip_rows['candidate'].unique().tolist()),
    'promotion_blocker': (
        'The selected action-cap candidate passes the two evaluated folds under tradable-window relative metrics, '
        'but isolated qlib fold backtests dropped pre-window warmup context. Promotion requires a warm-started official qlib '
        'backtest or an explicitly adopted tradable-window gate, plus fixed-split and wf_2020 validation.'
    ),
    'next_mechanism': 'warm_start_official_qlib_backtest_then_fixed_split_wf2020',
    'sidecars': [
        'metric_reconciliation.csv',
        'candidate_reconciliation_summary.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/1712.06924',
        'https://arxiv.org/abs/2006.04779',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    **selection,
    'source_selection_decision': source_selection.get('decision'),
    'lookback': LOOKBACK,
    'fold_windows': FOLD_WINDOWS,
    'reconciliation_rows': int(len(recon)),
    'selected_rows': selected_recon.to_dict('records'),
    'candidate_summary': candidate_summary.to_dict('records'),
}

(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2, sort_keys=True) + chr(10), encoding='utf-8')
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2, sort_keys=True) + chr(10), encoding='utf-8')
selection


{'experiment': 'metric_source_reconciliation_v1',
 'created_at': '2026-05-26T03:38:34.523825+00:00',
 'role': 'diagnose',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'sequence_risk_action_cap_screen_v1',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'target_relative_return_pp': 30.0,
 'target_max_dd_pct': 15.0,
 'decision': 'metric_source_reconciled_warmup_alignment_no_promotion',
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'root_cause': 'warmup_benchmark_alignment_gate_flip',
 'selected_candidate': 'seq_cap10_two_of_three',
 'selected_calendar_full_gate_all': False,
 'selected_tradable_full_gate_all': True,
 'selected_min_calendar_relative_return': -5.455231780988868,
 'selected_min_tradable_relative_return': 34.140849442349364,
 'selected_max_max_dd': 13.472617267038176,
 'selected_target_cap_changed_days_total': 104,
 'gate_flip_count': 5,
 'gate_flip_candidates': ['baseline_no_action_cap',
  'seq_cap1

In [6]:
selected_lines = []
for row in selected_recon.to_dict('records'):
    selected_lines.append(
        f"- {row['fold']}: calendar rel `{row['calendar_relative_return']:.2f}pp`, "
        f"tradable rel `{row['tradable_relative_return']:.2f}pp`, MaxDD `{row['max_dd']:.2f}%`, "
        f"full-gate flip `{bool(row['full_gate_flip'])}`, first tradable `{row['first_tradable_date']}`."
    )

conclusion = f"""
## Conclusion

Decision: `{selection['decision']}`. Promotion remains blocked.

Root cause: isolated qlib fold backtests in the source card start the portfolio at the fold boundary, but the policy cannot trade for the first `LOOKBACK={LOOKBACK}` rows. The source card's relative-return field matches the calendar benchmark, while the existing `evaluate_split`-style gate uses a tradable-window benchmark that starts after lookback. That benchmark-window difference creates the observed metric mismatch.

Selected candidate `{SELECTED_CANDIDATE}`:
{chr(10).join(selected_lines)}

Interpretation: under tradable-window relative metrics, `{SELECTED_CANDIDATE}` clears the two evaluated folds with max MaxDD `{selected_summary['max_max_dd']:.2f}%` and min tradable relative `{selected_summary['min_tradable_relative_return']:.2f}pp`. Under calendar relative metrics it fails because `wf_2023_2024` includes the warmup period's buy-and-hold move while the strategy is forced flat by the sliced frame. This is a harness metric-alignment issue, not evidence that a new reward or action-cap threshold should be tuned.

Next action: `{selection['next_mechanism']}`. Do not promote or retain an actor from this card; first rerun the selected action cap with warm-started official qlib evaluation, or explicitly adopt a tradable-window relative gate, then test fixed-split plus `wf_2020` / `wf_2022` / `wf_2023_2024` together.

Artifacts: `metric_reconciliation.csv`, `candidate_reconciliation_summary.csv`, `selection.json`, `summary.json`.
""".strip()

display(Markdown(conclusion))


## Conclusion

Decision: `metric_source_reconciled_warmup_alignment_no_promotion`. Promotion remains blocked.

Root cause: isolated qlib fold backtests in the source card start the portfolio at the fold boundary, but the policy cannot trade for the first `LOOKBACK=60` rows. The source card's relative-return field matches the calendar benchmark, while the existing `evaluate_split`-style gate uses a tradable-window benchmark that starts after lookback. That benchmark-window difference creates the observed metric mismatch.

Selected candidate `seq_cap10_two_of_three`:
- wf_2022: calendar rel `78.86pp`, tradable rel `68.89pp`, MaxDD `11.53%`, full-gate flip `False`, first tradable `2022-03-30`.
- wf_2023_2024: calendar rel `-5.46pp`, tradable rel `34.14pp`, MaxDD `13.47%`, full-gate flip `True`, first tradable `2024-03-28`.

Interpretation: under tradable-window relative metrics, `seq_cap10_two_of_three` clears the two evaluated folds with max MaxDD `13.47%` and min tradable relative `34.14pp`. Under calendar relative metrics it fails because `wf_2023_2024` includes the warmup period's buy-and-hold move while the strategy is forced flat by the sliced frame. This is a harness metric-alignment issue, not evidence that a new reward or action-cap threshold should be tuned.

Next action: `warm_start_official_qlib_backtest_then_fixed_split_wf2020`. Do not promote or retain an actor from this card; first rerun the selected action cap with warm-started official qlib evaluation, or explicitly adopt a tradable-window relative gate, then test fixed-split plus `wf_2020` / `wf_2022` / `wf_2023_2024` together.

Artifacts: `metric_reconciliation.csv`, `candidate_reconciliation_summary.csv`, `selection.json`, `summary.json`.